# TIL: Async/Await in Python — It's About Politeness, Not Speed

Today I sat down to actually *understand* Python's `async`/`await`, and it finally clicked. Here's the journey.

## An async function doesn't run when you call it

I defined a simple async function that pretends to do some slow work:

```python
import asyncio, time

async def slow_task(name, seconds):
    print(f"  {name}: started")
    await asyncio.sleep(seconds)
    print(f"  {name}: done after {seconds}s")
    return f"{name} result"
```

Then I called it:

```python
>>> slow_task("A", 2)
<coroutine object slow_task at 0x7a52be3a76a0>
```

No output. No "started". Just a coroutine object sitting there. Calling an async function doesn't *run* it — it creates a coroutine that's waiting to be driven by something.

## `await` runs it, but sequentially

```python
start = time.time()

r1 = await slow_task("A", 5)
r2 = await slow_task("B", 2)

print(f"Total time: {time.time() - start:.1f}s")
```

```
  A: started
  A: done after 5s
  B: started
  B: done after 2s

Total time: 7.0s
```

It works, but B doesn't start until A finishes. That's 5 + 2 = 7 seconds. We're being polite enough to *use* await, but we're still standing in a queue.

## `asyncio.gather` runs them concurrently

```python
start = time.time()

r1, r2 = await asyncio.gather(
    slow_task("A", 5),
    slow_task("B", 2)
)

print(f"Total time: {time.time() - start:.1f}s")
```

```
  A: started
  B: started
  B: done after 2s
  A: done after 5s

Total time: 5.0s
```

Both tasks start immediately. B finishes first, A finishes after 5 seconds. Total wall time: 5 seconds — the length of the *longest* task, not the sum. `gather` says "start all of these, and give me all the results when they're done."

## The key insight: `await` is about *yielding*, not waiting

This is where it really clicked for me. I made two versions of the same task:

```python
async def polite_task(name, seconds):
    print(f"  {name}: started")
    await asyncio.sleep(seconds)   # "I'll wait, go help others"
    print(f"  {name}: done after {seconds}s")
    return f"{name} result"

async def blocking_task(name, seconds):
    print(f"  {name}: started")
    time.sleep(seconds)            # "ZZZzzz... nobody moves"
    print(f"  {name}: done after {seconds}s")
    return f"{name} result"
```

The only difference: `asyncio.sleep` vs `time.sleep`.

```
=== Polite waiter ===
  A: started
  B: started
  B: done after 2s
  A: done after 5s
Total: 5.0s

=== Sleeping waiter ===
  A: started
  A: done after 5s
  B: started
  B: done after 2s
Total: 7.0s
```

`asyncio.sleep` *yields control* back to the event loop — "I'm waiting, go run something else." `time.sleep` blocks the entire thread — nobody else gets to do anything.

This is concurrency, not parallelism. There's still only one thread. But when a task politely says "I'm idle, go ahead," other tasks get a turn. It's like a single cashier serving multiple customers who are still getting their wallets out — instead of staring at one person until they're done.

## The takeaway

- `async def` creates a function that *returns a coroutine* — it doesn't run it.
- `await` runs a coroutine, but waits for it to finish before moving on.
- `asyncio.gather` runs multiple coroutines concurrently.
- Concurrency only works if your tasks actually *yield* (use `await`-able operations). A blocking call inside an async function defeats the whole purpose.

In [ ]:
#| hide


import subprocess
from pathlib import Path
from dialoghelper import curr_dialog

async def deploy_notebook():
    nb_name = Path((await curr_dialog())['name']).name + '.ipynb'
    src = f'/app/data/{(await curr_dialog())["name"]}.ipynb'
    dst = '/app/data/publish/portfolio/static/'
    print(nb_name)
    
    # Copy notebook to static folder
    subprocess.run(['cp', src, dst])
    
    # Deploy with plash
    subprocess.run(['plash_deploy'], cwd='/app/data/publish/portfolio')

In [ ]:
#| hide
await deploy_notebook()